# Week 7 — Hands-On ML Solution Data Visualizations

**Project:** FinMark Corporation · **Team:** 3 · **Course:** Math for Algebra and Machine Learning (A3101)

This notebook extends the team's Milestone 1 EDA using finalized Week 3 cleaned datasets. It combines NumPy statistical calculations with Matplotlib and Seaborn charts, then prints data-specific interpretations alongside the results. It is exploratory: descriptive differences and correlations do not establish causation.

**Inputs:** `customer_demographics_cleaned.csv`, `customer_transactions_cleaned.csv`, and `social_media_interactions_cleaned.csv` from `Week3_Preprocessing_Submission/`.


## 1. Load libraries and finalized datasets

The path resolver supports running this notebook from the project folder or from its `Week7_Data_Visualizations` subfolder. In Colab, upload the project folder (or edit `PROJECT_DIR` to the mounted location).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

# Find the project directory without relying on a machine-specific absolute path.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p for p in _candidates if (p / "Week3_Preprocessing_Submission").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find Week3_Preprocessing_Submission. Set PROJECT_DIR to the project folder.")
DATA_DIR = PROJECT_DIR / "Week3_Preprocessing_Submission"

demographics = pd.read_csv(DATA_DIR / "customer_demographics_cleaned.csv")
transactions = pd.read_csv(DATA_DIR / "customer_transactions_cleaned.csv", parse_dates=["transaction_date"])
social = pd.read_csv(DATA_DIR / "social_media_interactions_cleaned.csv", parse_dates=["interaction_date"])
# Signup dates are parsed for optional cohort or tenure analysis.
demographics["signup_date"] = pd.to_datetime(demographics["signup_date"], errors="coerce")

datasets = {"Customer demographics": demographics, "Customer transactions": transactions, "Social media interactions": social}
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns | missing cells: {df.isna().sum().sum():,}")


## 2. Additional numerical analysis with NumPy

Mean, median, population standard deviation and variance, minimum, maximum, quartiles, and IQR are calculated for `age` and `amount`. Population formulas (`ddof=0`) describe the complete cleaned file in hand; use `ddof=1` if estimating a wider population from a sample. The zero transaction amounts are retained because Milestone 1 documents that some represent free transactions.

In [ ]:
def numeric_summary(series):
    x = series.dropna().to_numpy(dtype=float)
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    return {
        "n": x.size,
        "mean": np.mean(x),
        "median": median,
        "std_dev (population)": np.std(x, ddof=0),
        "variance (population)": np.var(x, ddof=0),
        "minimum": np.min(x),
        "Q1": q1,
        "Q3": q3,
        "maximum": np.max(x),
        "IQR": q3 - q1,
    }

numeric_stats = pd.DataFrame({
    "Customer age (years)": numeric_summary(demographics["age"]),
    "Transaction amount": numeric_summary(transactions["amount"]),
}).T

display(numeric_stats.round(2))
for label, row in numeric_stats.iterrows():
    skew_hint = "mean above median (right-skew tendency)" if row["mean"] > row["median"] else "mean below median (left-skew tendency)" if row["mean"] < row["median"] else "mean and median coincide"
    print(f"{label}: {skew_hint}; middle half spans {row['Q1']:.2f}–{row['Q3']:.2f}.")


### Outlier rule and interpretation

The 1.5 × IQR rule flags potential statistical outliers for review. A flag is not proof of a data error. In Milestone 1, neither age nor transaction amount had IQR outliers; this cell recomputes that check from the loaded files.

In [ ]:
for label, series in [("Age", demographics["age"]), ("Transaction amount", transactions["amount"])]:
    x = series.dropna().to_numpy(dtype=float)
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = x[(x < low) | (x > high)]
    print(f"{label}: {flagged.size:,} / {x.size:,} values outside [{low:.2f}, {high:.2f}] by the 1.5×IQR rule.")


## 3. Frequency distributions and numerical comparisons

Histograms show each variable's shape; box plots compare spread and make potential extremes visible. The age and amount axes use their native units.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(data=demographics, x="age", bins=np.arange(15, 76, 5), kde=True, ax=axes[0, 0], color="#4472C4")
axes[0, 0].set(title="Customer age distribution", xlabel="Age (years)", ylabel="Customers")
sns.boxplot(data=demographics, x="age", ax=axes[1, 0], color="#9DBBEA")
axes[1, 0].set(title="Age spread and potential extremes", xlabel="Age (years)")
sns.histplot(data=transactions, x="amount", bins=25, kde=True, ax=axes[0, 1], color="#ED7D31")
axes[0, 1].set(title="Transaction amount distribution", xlabel="Amount", ylabel="Transactions")
sns.boxplot(data=transactions, x="amount", ax=axes[1, 1], color="#F4B183")
axes[1, 1].set(title="Transaction amount spread and potential extremes", xlabel="Amount")
fig.suptitle("Numerical distributions in the cleaned datasets", fontsize=15)
fig.tight_layout()
plt.show()

print("Interpretation: compare the histogram peaks and tails with the mean/median table. The amount values span a wider numeric range; zero-valued records should be read in light of the documented free-transaction rule. The box plots visually complement, rather than replace, the IQR flag counts above.")


## 4. Category frequencies and imbalance

The count plots compare dominant categories across datasets. Sentiment categories are displayed in observed frequency order so that the low counts for `Very Positive` and `Very Negative` remain visible.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
plots = [
    (demographics, "gender", "Gender", axes[0, 0]),
    (transactions, "product_category", "Product category", axes[0, 1]),
    (social, "platform", "Social platform", axes[1, 0]),
    (social, "sentiment", "Sentiment", axes[1, 1]),
]
for df, col, title, ax in plots:
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color=sns.color_palette("deep")[0])
    ax.set(title=f"Frequency: {title}", xlabel="Records", ylabel=title)
fig.suptitle("Category frequencies", fontsize=15)
fig.tight_layout()
plt.show()

sentiment_share = social["sentiment"].value_counts(normalize=True).mul(100)
print("Sentiment percentages:")
display(sentiment_share.rename("percent").round(2).to_frame())
rare = sentiment_share[sentiment_share < 5]
if len(rare):
    print("Interpretation: rare sentiment class(es) under 5%:", ", ".join(f"{k} ({v:.2f}%)" for k, v in rare.items()), ". If sentiment is later a prediction target, evaluate per-class performance and class balance explicitly.")


## 5. Relationships and grouped comparisons

A box plot compares the transaction amount distribution by product category. The sentiment-by-platform heatmap uses **row percentages**, so each platform's composition sums to 100% and unequal platform totals do not dominate the color scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
category_order = transactions.groupby("product_category")["amount"].median().sort_values().index
sns.boxplot(data=transactions, x="product_category", y="amount", order=category_order, ax=axes[0], color="#9DBBEA")
axes[0].set(title="Transaction amounts by product category", xlabel="Product category", ylabel="Amount")
axes[0].tick_params(axis="x", rotation=25)

sentiment_by_platform = pd.crosstab(social["platform"], social["sentiment"], normalize="index").mul(100)
sns.heatmap(sentiment_by_platform, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "Within-platform percent"}, ax=axes[1])
axes[1].set(title="Sentiment composition by platform", xlabel="Sentiment", ylabel="Platform")
fig.tight_layout()
plt.show()

medians = transactions.groupby("product_category")["amount"].median().sort_values()
print(f"Median transaction amounts by category range from {medians.iloc[0]:.2f} ({medians.index[0]}) to {medians.iloc[-1]:.2f} ({medians.index[-1]}), a difference of {medians.iloc[-1] - medians.iloc[0]:.2f}.")
print("Interpretation: inspect overlap in the box plots before treating median differences as meaningful. The heatmap describes composition only; it does not show that platform causes sentiment.")


### Customer-level age and spending relationship

Transaction rows must be aggregated before joining to customer demographics. The merge below uses one row per customer, avoiding event-level many-to-many duplication. Results describe customers with both demographics and at least one transaction only.

In [ ]:
spend_by_customer = transactions.groupby("customer_id", as_index=False).agg(
    total_spend=("amount", "sum"), transaction_count=("transaction_id", "nunique")
)
age_spend = demographics[["customer_id", "age", "income_level"]].merge(spend_by_customer, on="customer_id", how="inner", validate="one_to_one")
r_age_spend = age_spend["age"].corr(age_spend["total_spend"])
print(f"Matched customers (demographics + transactions): {len(age_spend):,}")
print(f"Pearson correlation, age vs total spend: r = {r_age_spend:.3f}")

plt.figure(figsize=(9, 5))
sns.regplot(data=age_spend, x="age", y="total_spend", scatter_kws={"alpha": 0.35, "s": 25}, line_kws={"color": "crimson"}, lowess=True)
plt.title("Customer age and total recorded spend")
plt.xlabel("Age (years)")
plt.ylabel("Total spend across recorded transactions")
plt.tight_layout()
plt.show()
print("Interpretation: the fitted line and Pearson r summarize linear association in the matched sample; neither establishes causation, and non-linear patterns may not be captured by r.")


## 6. Trends over time

Monthly event counts are plotted separately because the source windows differ: transactions span July 2022–June 2024, while social interactions span July 2023–June 2024. The shaded social period makes that coverage difference explicit.

In [ ]:
tx_monthly = transactions.set_index("transaction_date").resample("MS").size().rename("Transactions")
social_monthly = social.set_index("interaction_date").resample("MS").size().rename("Social interactions")
fig, ax = plt.subplots(figsize=(13, 5))
tx_monthly.plot(ax=ax, marker="o", linewidth=1.8, label="Transactions")
social_monthly.plot(ax=ax, marker="s", linewidth=1.8, label="Social interactions")
ax.axvspan(social_monthly.index.min(), social_monthly.index.max(), color="gray", alpha=0.08, label="Social data coverage")
ax.set(title="Monthly transaction and social interaction counts", xlabel="Month", ylabel="Recorded events")
ax.legend(title="Series / coverage")
fig.tight_layout()
plt.show()

print(f"Transactions: {tx_monthly.min():.0f}–{tx_monthly.max():.0f} per month; peak {tx_monthly.idxmax():%b %Y} ({tx_monthly.max():.0f}).")
print(f"Social interactions: {social_monthly.min():.0f}–{social_monthly.max():.0f} per month; peak {social_monthly.idxmax():%b %Y} ({social_monthly.max():.0f}).")
print("Interpretation: monthly fluctuations identify busy and quieter periods. Two years of transaction counts and one year of social counts do not by themselves establish recurring seasonality; compare aligned months for cross-channel comparisons.")


## 7. Findings for Milestone 2 and future ML work

Use the computed tables and charts to report the current dataset's numerical summaries, category balance, group differences, monthly highs/lows, and matched-customer relationship. Milestone 1 offers useful expectations to verify against this run: age and transaction amount had no 1.5 × IQR outliers; most major categories were fairly balanced; extreme sentiment classes were near 1% each; category median differences were modest; and age versus total spend had near-zero linear correlation in the matched sample.

**Modeling implications:** retain the zero-amount meaning in feature documentation; treat identifiers as join keys rather than ordinary predictors; report matched population sizes for every cross-dataset result; consider class-aware evaluation if predicting sentiment; and define a business target before selecting a model. Location has very high cardinality and requires investigation before encoding.

**Cautions:** charts and statistics are descriptive, records cover different time windows, the cleaned files do not contain identical customer populations, and associations do not imply causation. Re-run all cells from top to bottom after changing the input data.